# Análise de Eficiência Operacional — Concessão de Crédito

## Objetivo do Projeto

Este projeto simula um sistema completo de análise de crédito e eficiência operacional bancária, incluindo:

- **Modelo de Machine Learning** para prever probabilidade de inadimplência
- **Sistema de decisão automática** (aprovação / análise manual / reprovação)
- **Priorização de fila** de análise manual por valor e perfil
- **Simulação operacional** para identificar gargalos, backlog e tempo de espera
- **Dashboard Power BI** para visualização gerencial
- **Aplicação web** com chatbot para apoio a analistas

**Stack:** Python, SQL Server, Power BI, Streamlit


##  Contexto de Negócio

### O que é Concessão de Crédito?

É o processo pelo qual o banco decide se vai emprestar dinheiro para um cliente e em quais condições. Quando alguém pede um empréstimo pessoal ou crédito consignado, o banco avalia o **risco de inadimplência** (chance do cliente não pagar).

### Fluxo típico do processo
```
Cliente solicita crédito
        ↓
1. Recepção → Pedido entra no sistema
        ↓
2. Verificação documental → Documentos corretos?
        ↓
3. Análise de crédito → Score, renda, histórico
        ↓
4. Decisão → Aprovado / Negado / Análise manual
        ↓
5. Formalização → Contrato e assinaturas
        ↓
6. Liberação → Dinheiro na conta
```

### Por que isso importa para o banco?

| Impacto | Consequência |
|---------|--------------|
| **Lucro** | Processo lento = cliente vai para concorrência = receita perdida |
| **Risco** | Processo ruim pode aprovar maus pagadores ou negar bons clientes |
| **Custo** | Análise manual custa tempo e dinheiro (salário de analistas) |
| **Imagem** | Experiência ruim gera reclamações e prejudica reputação |

##  Problema e Perguntas de Negócio

### Os 3 Gargalos que este projeto analisa

| Gargalo | Descrição | Impacto |
|---------|-----------|---------|
| **Dependência Manual** | Análises que dependem de humanos causam atrasos e erros | Lentidão, retrabalho, custo alto |
| **SLA Estourado** | Prazos de resposta ao cliente não são cumpridos | Cliente insatisfeito, perda para concorrência |
| **Custos Ocultos** | Tempo e erros geram custos que não são visíveis | Dificuldade de justificar investimentos |

### Perguntas que este projeto responde

1. **Qual a probabilidade de um cliente ser inadimplente?** → Modelo de ML
2. **Quais clientes podem ser aprovados/reprovados automaticamente?** → Sistema de decisão
3. **Quem deve ser priorizado na fila de análise manual?** → Sistema de priorização
4. **Qual o tamanho do backlog e tempo médio de espera?** → Simulação operacional
5. **Onde estão os gargalos do processo?** → Dashboard e chatbot

##  Estrutura dos Dados

Para este projeto, vamos gerar dados simulados que representam clientes solicitando crédito.

### Tabela: `solicitacoes_credito`

| Coluna | Tipo | Descrição |
|--------|------|-----------|
| `id` | int | Identificador único da solicitação |
| `data_solicitacao` | datetime | Data/hora do pedido |
| `idade` | int | Idade do cliente |
| `renda_mensal` | float | Renda mensal declarada (R$) |
| `valor_solicitado` | float | Valor do empréstimo pedido (R$) |
| `prazo_meses` | int | Prazo de pagamento desejado |
| `score_credito` | int | Score de crédito (0-1000) |
| `tem_imovel` | bool | Cliente possui imóvel próprio? |
| `tem_veiculo` | bool | Cliente possui veículo? |
| `tempo_emprego_anos` | float | Tempo no emprego atual |
| `qtd_emprestimos_ativos` | int | Quantos empréstimos já possui |
| `historico_inadimplencia` | bool | Já ficou inadimplente antes? |
| `tipo_credito` | str | 'pessoal' ou 'consignado' |
| `inadimplente` | bool | **TARGET:** Cliente não pagou? (para treinar o modelo) |

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

from sqlalchemy import create_engine 
from dotenv import load_dotenv 
import os 


In [8]:
load_dotenv()

server = os.getenv ('SQL_SERVER')
database = os.getenv('SQL_DATABASE')

connection_string = f"mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"

engine = create_engine(connection_string)

try:
    with engine.connect() as conn:
        print(f" Conectado com sucesso ao banco: {database}")
        print (f" Servidor: {server}")
except Exception as e:
    print(f" Erro na conexão: {e}")


 Conectado com sucesso ao banco: CreditoOperacional
 Servidor: R3G1N4LD0
